In [ ]:
%pip install pandas
%pip install numpy
%pip install matplotlib
%pip install seaborn
%pip install statsmodels
%pip install scikit-learn
%pip install prophet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn
from statsmodels.tsa.seasonal import STL

In [ ]:
import os

file_path = "../data/processed/Delay_rate_by_month.csv"

if not os.path.exists(file_path):
    print(f"Error: The File '{file_path}' does not exist ")
else:
    try:
        df = pd.read_csv(
            file_path,
            on_bad_lines ="skip",
            encoding = 'utf-8'
        )
        print("CSV loaded successfully")
        print(df.head())
    except pd.errors.EmptyDataError:
        print("Error: The CSV is empty")
    except pd.errors.ParserError as e:
        print(f"Error: Parse error (structrally corrupt CSV) {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [ ]:
plt.Figure(figsize=(12,8))
plt.plot(df['month'],df['delivered_orders'],color='blue',linestyle='-',label='Delivered Orders')
plt.plot(df['month'],df['delayed_orders'],color='red',linestyle='-',label='Delayed Orders')
plt.title('US e-commence Delivered Orders vs Delayed Orders 2019-2025')
plt.xlabel('Month',fontsize = 12)
plt.legend()


plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12,8))
plt.plot(df['month'],df['delay_rate'],marker='o', linewidth=2, label='Delay Rate')

plt.title('US e-commence Delay Rate 2019-2025',fontsize=14)
plt.xlabel('Month',fontsize=12)
plt.ylabel('Delay Rate (%)',fontsize=12)
plt.legend()

plt.show()

In [ ]:
df = df.copy()
df['month'] = pd.to_datetime(df['month'],format='%Y-%m')
df.set_index('month',inplace=True)
res = STL(df['delay_rate'],period=12, robust = True).fit()

residuals = res.resid

z_scores = (residuals - residuals.mean())/residuals.std()

outliers = df[np.abs(z_scores) > 3]
print('Confirmed Outliers :')
print(outliers)

In [ ]:
res.plot()
plt.show()

In [ ]:
# Last data point is highly the glitch or error. Due to the features of the STL model, it cannot detect the outlier at the end of the dataset
df.iloc[-1, df.columns.get_loc('delay_rate')] = df.iloc[-13]['delay_rate']


In [ ]:
plt.figure(figsize=(12,8))
plt.plot(df['month'],df['delay_rate'],marker='o', linewidth=2, label='Delay Rate')

plt.title('US e-commence Delay Rate 2019-2025',fontsize=14)
plt.xlabel('Month',fontsize=12)
plt.ylabel('Delay Rate (%)',fontsize=12)
plt.legend()

plt.show()

In [ ]:
res = STL(df['delay_rate'],period=12, robust = True).fit()

residuals = res.resid

z_scores = (residuals - residuals.mean())/residuals.std()

outliers = df[np.abs(z_scores) > 3]
print('Confirmed Outliers :')
print(outliers)

In [ ]:
res.plot()
plt.show()

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
result = seasonal_decompose(df['delay_rate'], model='multiplicative',extrapolate_trend='freq')
result.plot()
plt.suptitle('Seasonal Decomposition of delay rate time series')
plt.tight_layout()
plt.show()

Visually confirmed the seaonsonality

In [ ]:
#Display Seasonlity graph
plt.figure(figsize=(8,4))
plt.plot(result.seasonal,label='seasonal component')
plt.title('Seasonal Component of Delay Rate')
plt.xlabel('Year')
plt.ylabel('Seasonal Component')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
#plot the original data and origin data without the seasonal component
plt.figure(figsize=(8,6))
plt.plot(df['delay_rate'],label="Original Time Series",color='blue')
data_without_seasonal = df['delay_rate'] / result.seasonal
plt.plot(data_without_seasonal,label="Original Data without Seasonal Component",color='green')
plt.title('Delay Rate Time Series with and without Seasonality')
plt.xlabel('Year')
plt.ylabel('Delay Rate(%)')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
from statsmodels.tsa.stattools import adfuller

adf_result = adfuller(data_without_seasonal)

print('ADF statistic:', adf_result[0])
print('p-value', adf_result[1])

if adf_result[1] < 0.05:
    print("The data is stationary (p-value < 0.05)")
else:
    print('The data is not stationary (p-value >= 0.05)')

In [ ]:
#find p and q
from statsmodels.graphics.tsaplots import plot_acf,plot_pacf

plot_acf(data_without_seasonal,lags=12)
plot_pacf(data_without_seasonal,lags=12)
plt.show()

In [ ]:
split_idx = int(len(data_without_seasonal) * 0.8)
train = data_without_seasonal.iloc[:split_idx]
test = data_without_seasonal.iloc[split_idx:]


In [ ]:
#Visually confirmed p = 1 and q = 1
#fit ARMA model
from statsmodels.tsa.arima.model import ARIMA
p = 1
d = 0
q = 1
arma = ARIMA(train, order=(p,d,q))
arma_fit = arma.fit()

print(arma_fit.summary())

In [ ]:
arma_fit.plot_diagnostics(figsize=(10,8))
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import mean_absolute_error,root_mean_squared_error,mean_absolute_percentage_error

forecast = arma_fit.forecast(steps=len(test))

mae = mean_absolute_error(test,forecast)
rmse = root_mean_squared_error(test,forecast)
mape = mean_absolute_percentage_error(test,forecast)

print(f'Mean absolute error(MAE): {mae:.4f}')
print(f'Root mean squared error (RMSE):{rmse:.4f}')
print(f'Mean Absolute Percentage of Error (MAPE): {mape * 100:.2f}%')

In [ ]:
plt.figure(figsize=(10,6))
plt.plot(train.index, train, label = "Train data")
plt.plot(test.index, test, label="Actual Test data", color="blue")
plt.plot(test.index,forecast,label="ARMA forecast",color="red",linestyle="--")
plt.title(f"ARMA({p},{q}) Forecast VS Actuals")
plt.legend()
plt.tight_layout()
plt.show()
